# 🚀 Stack 2.9 - Kaggle Training Notebook

**Free GPU training on Kaggle**

This notebook trains a LoRA adapter for Stack 2.9 on **Qwen2.5-Coder-7B** using Kaggle's free GPU.

⏱️ **Expected runtime:** 2-4 hours
💾 **VRAM needed:** ~16GB (Kaggle P100 has 16GB)

---

**Instructions:**
1. Enable GPU: Settings → Accelerator → GPU P100
2. Run cells in order from the top
3. Model auto-downloads if not present

---

In [ ]:
# STEP 1: Check GPU
!nvidia-smi

In [ ]:
# STEP 2: Clone repo
import os
import shutil

REPO_DIR = "/kaggle/working/stack-2.9"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone https://github.com/my-ai-stack/stack-2.9.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"✅ Working in: {os.getcwd()}")

In [ ]:
# STEP 3: Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers peft accelerate datasets pyyaml tqdm scipy bitsandbytes
print("✅ Dependencies installed")

In [ ]:
# STEP 4: Setup paths (MODEL_DIR, OUTPUT_DIR)
import os

REPO_DIR = "/kaggle/working/stack-2.9"
MODEL_DIR = os.path.join(REPO_DIR, "base_model_qwen7b")
OUTPUT_DIR = os.path.join(REPO_DIR, "training_output")

print(f"REPO_DIR: {REPO_DIR}")
print(f"MODEL_DIR: {MODEL_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

In [ ]:
# STEP 5: Download model (if not exists)
from transformers import AutoModelForCausalLM, AutoTokenizer

if os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("✅ Model already exists, skipping download!")
else:
    print("⬇️ Downloading model (Qwen2.5-Coder-7B)...")
    print("This takes ~10-15 minutes...")
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B", trust_remote_code=True)
    tokenizer.save_pretrained(MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-7B", trust_remote_code=True)
    model.save_pretrained(MODEL_DIR)
    print("✅ Model downloaded!")

!ls -lh {MODEL_DIR} | head -5

In [ ]:
# STEP 6: Create config
import yaml
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

config = {
    'model': {'name': MODEL_DIR, 'trust_remote_code': True, 'torch_dtype': 'float16'},
    'data': {'input_path': './data/final/train.jsonl', 'max_length': 2048},
    'lora': {'r': 16, 'alpha': 32, 'dropout': 0.05,
             'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
             'bias': 'none', 'task_type': 'CAUSAL_LM'},
    'training': {'num_epochs': 1, 'batch_size': 2, 'gradient_accumulation': 4,
                 'learning_rate': 2e-4, 'warmup_steps': 50, 'weight_decay': 0.01,
                 'max_grad_norm': 1.0, 'logging_steps': 5, 'eval_steps': 100,
                 'save_steps': 200, 'save_total_limit': 2, 'fp16': True, 'bf16': False,
                 'gradient_checkpointing': True},
    'output': {'lora_dir': os.path.join(OUTPUT_DIR, 'lora'),
               'merged_dir': os.path.join(OUTPUT_DIR, 'merged')},
    'quantization': {'enabled': False},
    'hardware': {'device': 'cuda', 'num_gpus': 1, 'use_4bit': False, 'use_8bit': False}
}

config_path = os.path.join(OUTPUT_DIR, "train_config.yaml")
with open(config_path, 'w') as f:
    yaml.dump(config, f)

print(f"✅ Config saved to: {config_path}")
print(f"   Device: {config['hardware']['device']}")

In [ ]:
# STEP 7: Train LoRA
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack/training"))

print("="*60)
print("STARTING TRAINING")
print("="*60)

from train_lora import train_lora
trainer = train_lora(config_path)

print("="*60)
print("TRAINING COMPLETED!")
print("="*60)

In [ ]:
# STEP 8: Merge model
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "stack/training"))
from merge_adapter import merge_adapter

merged_dir = os.path.join(OUTPUT_DIR, "merged")
os.makedirs(merged_dir, exist_ok=True)

merge_config = {
    'model': {'name': MODEL_DIR, 'trust_remote_code': True},
    'output': {'lora_dir': os.path.join(OUTPUT_DIR, 'lora'), 'merged_dir': merged_dir},
    'quantization': {'enabled': False}
}

merge_cfg_path = os.path.join(OUTPUT_DIR, "merge_config.yaml")
with open(merge_cfg_path, 'w') as f:
    yaml.dump(merge_config, f)

merge_adapter(merge_cfg_path, os.path.join(OUTPUT_DIR, "lora"), merged_dir)

print(f"✅ Merged model saved to: {merged_dir}")
!ls -lh {merged_dir}

In [ ]:
# STEP 9: Done!
print("="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print(f"LoRA adapter: {os.path.join(OUTPUT_DIR, 'lora')}")
print(f"Merged model: {os.path.join(OUTPUT_DIR, 'merged')}")
print("\n📥 Download from: Kaggle → Output tab")